# Curadoria automatizada de genes de esporulação e membrana externa/LPS

Este notebook reconstrói, de forma reprodutível, uma matriz de presença/ausência para **61 marcadores em 229 genomas bacterianos**:

- **15 marcadores de membrana externa/LPS**
- **46 marcadores de esporulação**

A estratégia é única em todo o notebook:

1. **Evidência direta do Bakta**: presença quando o campo `Gene` corresponde **exatamente** ao nome do marcador.
2. **Evidência por OrthoFinder**: os orthogroups candidatos são descobertos a partir das proteínas anotadas pelo Bakta. Um par marcador–orthogroup é aprovado automaticamente somente quando:
   - `n_seed / n_proteins >= 0.50`; e
   - `n_seed >= 2`.
3. **Chamada híbrida**: `Bakta positivo OU pertencimento a orthogroup aprovado`.

Os critérios de orthogroup são **heurísticas de concordância de anotação**, não validação funcional definitiva. Pertencer ao mesmo orthogroup não demonstra por si só equivalência funcional, e a presença de genes de esporulação ou de membrana externa/LPS não deve ser interpretada automaticamente como capacidade fenotípica de esporular ou como uma arquitetura de envelope completa.

### Regra para ausências

Um `0` só é emitido quando a anotação do Bakta carece da presença do gene em questão, portanto não encontramos o equivalente apontado pelo OrthoFinder, logo, nenhum dos dois métodos gerou evidência positiva. Se houver cobertura Bakta incompleta, positivos podem ser preservados, mas células sem evidência positiva permanecem `NA`.

Assim, o notebook **não transforma silenciosamente dados não avaliados em ausência**.


### Resultados do notebook

Uma execução válida exporta **cinco tabelas**: matriz binária, matriz para heatmap, revisão automática dos orthogroups, resumo por marcador e novas presenças.

As tabelas extensas de proteínas e verificações ficam em memória.

**Metadados externos, preservados sem inferência:**
- `Spore former`: classificação de esporulação de **Galperin**
  (`1` esporulante, `2` não esporulante, `0` sem classificação);
- `expected`: informação sobre o **tipo de envelope celular, proveniente da literatura**
  (por exemplo, `diderm` ou `monoderm`);

Nenhuma dessas categorias é deduzida automaticamente dos 61 marcadores neste notebook.

## 1. Configuração de caminhos e parâmetros

In [ ]:

from pathlib import Path
import re
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 160)

# ---------------------------------------------------------------------
# Caminhos: altere somente esta seção quando mover os arquivos.
# ---------------------------------------------------------------------
def encontrar_raiz():
    pasta = Path.cwd().resolve()
    for candidato in (pasta, *pasta.parents):
        if (candidato / "atividade").is_dir() and (candidato / "preparacao").is_dir():
            return candidato
    raise FileNotFoundError("Abra o notebook na pasta do projeto ou em preparacao/.")

PROJECT = encontrar_raiz()
DATA_DIR = PROJECT / "preparacao" / "dados_brutos"
OUTPUT_DIR = PROJECT / "preparacao" / "resultados_curadoria_61"
# Os dados brutos acompanham apenas Bacillota_projeto_completo_revisado.zip.


HISTORICAL_MATRIX_PATH = DATA_DIR / "heatmap_matrix2_spore_categories.tsv"
ORTHOGROUPS_PATH = DATA_DIR / "Orthogroups.tsv"
BAKTA_TSV_PATH = DATA_DIR / "baktatsv.tsv"

# Opcional: matriz produzida em uma execução anterior, usada apenas para comparação.
PREVIOUS_FINAL_PATH = DATA_DIR / "gene_presence_absence_final.tsv"

# ---------------------------------------------------------------------
# Parâmetros metodológicos fixados para esta curadoria.
# ---------------------------------------------------------------------
MIN_SEED_FRACTION = 0.50
MIN_SEEDS = 2

EXPECTED_N_ASSEMBLIES = 229
EXPECTED_N_OLD_MARKERS = 58
EXPECTED_N_MEMBRANE_OLD = 12
EXPECTED_N_SPORULATION = 46
EXPECTED_N_MARKERS = 61

AUX_COLUMNS = [
    "label",
    "assembly",
    "Spore former",
    "n_om",
    "observed",
    "expected",
]

# Recuperados da estratégia anterior e validados contra a matriz histórica.
OLD_MEMBRANE_MARKERS = [
    "fabZ", "kdsA", "kdsB", "kdsC", "kdsD", "lptB",
    "lpxA", "lpxB", "lpxC", "lpxD", "lpxK", "waaA",
]

NEW_MEMBRANE_MARKERS = ["bamA", "lptA", "ompH"]

# Valores históricos fornecidos como referência de comparação.
# Eles NÃO são usados para forçar chamadas.
REFERENCE_DIRECT_COUNTS = {
    "bamA": 28,
    "lptA": 11,
    "ompH": 21,
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Diretório de dados:", DATA_DIR.resolve())
print("Diretório de resultados:", OUTPUT_DIR.resolve())



## 2. Carregamento e validação dos arquivos

A matriz histórica define o universo de 229 assemblies, os 58 marcadores anteriores, a ordem histórica e as seis colunas auxiliares. Ela **não é usada como evidência positiva ou negativa na nova chamada híbrida**; serve como fonte de metadados e para comparação.

O arquivo do OrthoFinder é validado antes de qualquer transformação. Os identificadores verdadeiros `OG...` são preservados; não são gerados IDs provisórios.

O arquivo consolidado do Bakta é lido em blocos para reduzir o uso de memória e apenas as assemblies do conjunto didático são mantidas em memória.


In [ ]:

def require_file(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(
            f"{description} não encontrado em: {path.resolve()}\n"
            "Ajuste os caminhos na célula de configuração."
        )

require_file(HISTORICAL_MATRIX_PATH, "Matriz histórica")
require_file(ORTHOGROUPS_PATH, "Matriz Orthogroups do OrthoFinder")
require_file(BAKTA_TSV_PATH, "Tabela consolidada do Bakta")

historical = pd.read_csv(HISTORICAL_MATRIX_PATH, sep="\t")

missing_aux = [c for c in AUX_COLUMNS if c not in historical.columns]
if missing_aux:
    raise ValueError(f"Colunas auxiliares ausentes na matriz histórica: {missing_aux}")

if historical["assembly"].isna().any():
    raise ValueError("A matriz histórica contém assembly ausente.")
if historical["assembly"].duplicated().any():
    dup = historical.loc[historical["assembly"].duplicated(keep=False), "assembly"].tolist()
    raise ValueError(f"Assemblies duplicadas na matriz histórica: {dup[:20]}")

historical["assembly"] = historical["assembly"].astype("string").str.strip()
target_assemblies = historical["assembly"].tolist()
target_set = set(target_assemblies)

if len(target_assemblies) != EXPECTED_N_ASSEMBLIES:
    raise ValueError(
        f"Esperadas {EXPECTED_N_ASSEMBLIES} assemblies; "
        f"a matriz histórica contém {len(target_assemblies)}."
    )

historical_markers = [c for c in historical.columns if c not in AUX_COLUMNS]
if len(historical_markers) != EXPECTED_N_OLD_MARKERS:
    raise ValueError(
        f"Esperados {EXPECTED_N_OLD_MARKERS} marcadores históricos; "
        f"foram encontrados {len(historical_markers)}."
    )

missing_old_membrane = [g for g in OLD_MEMBRANE_MARKERS if g not in historical_markers]
if missing_old_membrane:
    raise ValueError(
        "Marcadores históricos de membrana esperados não foram encontrados: "
        f"{missing_old_membrane}"
    )

sporulation_markers = [
    c for c in historical_markers
    if c not in OLD_MEMBRANE_MARKERS
]

if len(sporulation_markers) != EXPECTED_N_SPORULATION:
    raise ValueError(
        f"Esperados {EXPECTED_N_SPORULATION} marcadores de esporulação; "
        f"foram encontrados {len(sporulation_markers)}."
    )

# Preserva a ordem relativa histórica; os três marcadores novos entram ao fim
# do bloco de membrana externa/LPS.
markers = OLD_MEMBRANE_MARKERS + NEW_MEMBRANE_MARKERS + sporulation_markers
marker_category = {
    **{g: "outer_membrane_LPS" for g in OLD_MEMBRANE_MARKERS + NEW_MEMBRANE_MARKERS},
    **{g: "sporulation" for g in sporulation_markers},
}

if len(markers) != EXPECTED_N_MARKERS or len(set(markers)) != EXPECTED_N_MARKERS:
    raise ValueError("A lista final de marcadores não contém 61 nomes únicos.")

# Validação dos estados históricos.
bad_historical_values = {}
for marker in historical_markers:
    values = set(historical[marker].dropna().unique().tolist())
    if not values.issubset({0, 1}):
        bad_historical_values[marker] = sorted(values, key=str)
if bad_historical_values:
    raise ValueError(
        "Marcadores históricos com valores diferentes de 0/1: "
        f"{bad_historical_values}"
    )

spore_categories = set(historical["Spore former"].dropna().unique().tolist())
if not spore_categories.issubset({0, 1, 2}):
    raise ValueError(
        "`Spore former` deve usar apenas as categorias 0, 1 e 2. "
        f"Valores encontrados: {sorted(spore_categories, key=str)}"
    )

# -----------------------------
# OrthoFinder
# -----------------------------
orthogroups_wide = pd.read_csv(ORTHOGROUPS_PATH, sep="\t", dtype="string")
if orthogroups_wide.shape[1] < 2:
    raise ValueError("Orthogroups.tsv não possui colunas de genomas.")

orthogroup_id_col = orthogroups_wide.columns[0]
orthogroup_ids = orthogroups_wide[orthogroup_id_col].astype("string").str.strip()

if orthogroup_ids.isna().any() or orthogroup_ids.duplicated().any():
    raise ValueError("Os identificadores de orthogroup estão ausentes ou duplicados.")

if not orthogroup_ids.str.fullmatch(r"OG\d+").all():
    examples = orthogroup_ids[~orthogroup_ids.str.fullmatch(r"OG\d+")].head(10).tolist()
    raise ValueError(
        "A primeira coluna não contém identificadores OG válidos em todas as linhas. "
        f"Exemplos problemáticos: {examples}"
    )

orthogroups_wide = orthogroups_wide.rename(columns={orthogroup_id_col: "orthogroup"})
orthogroups_wide["orthogroup"] = orthogroup_ids

of_assembly_columns = list(orthogroups_wide.columns[1:])
if len(of_assembly_columns) != len(set(of_assembly_columns)):
    raise ValueError("Há colunas de assembly duplicadas em Orthogroups.tsv.")

missing_in_of = sorted(target_set - set(of_assembly_columns))
extra_in_of = sorted(set(of_assembly_columns) - target_set)

if missing_in_of or extra_in_of:
    raise ValueError(
        "As colunas do OrthoFinder não correspondem exatamente ao universo histórico.\n"
        f"Ausentes no OrthoFinder: {missing_in_of[:20]}\n"
        f"Extras no OrthoFinder: {extra_in_of[:20]}\n"
        "Nenhuma correspondência aproximada foi aplicada automaticamente."
    )

# -----------------------------
# Bakta
# -----------------------------
bakta_header = pd.read_csv(BAKTA_TSV_PATH, sep="\t", nrows=0)
required_bakta_columns = ["Type", "Locus Tag", "Gene", "Product", "assembly"]
missing_bakta_columns = [c for c in required_bakta_columns if c not in bakta_header.columns]
if missing_bakta_columns:
    raise ValueError(
        f"Colunas necessárias ausentes em baktatsv.tsv: {missing_bakta_columns}"
    )

bakta_chunks = []
all_bakta_assemblies = set()

for chunk in pd.read_csv(
    BAKTA_TSV_PATH,
    sep="\t",
    usecols=required_bakta_columns,
    dtype="string",
    chunksize=250_000,
):
    chunk["assembly"] = chunk["assembly"].str.strip()
    chunk["Locus Tag"] = chunk["Locus Tag"].str.strip()
    chunk["Gene"] = chunk["Gene"].str.strip()
    all_bakta_assemblies.update(chunk["assembly"].dropna().tolist())
    bakta_chunks.append(chunk.loc[chunk["assembly"].isin(target_set)].copy())

bakta = pd.concat(bakta_chunks, ignore_index=True) if bakta_chunks else pd.DataFrame(
    columns=required_bakta_columns
)

if bakta.empty:
    raise ValueError(
        "Nenhuma linha do Bakta corresponde às 229 assemblies da matriz histórica."
    )

# Uma chave assembly + Locus Tag ambígua inviabiliza o merge seguro.
bakta_with_locus = bakta.dropna(subset=["Locus Tag"]).copy()
duplicated_keys = bakta_with_locus.duplicated(["assembly", "Locus Tag"], keep=False)
if duplicated_keys.any():
    examples = (
        bakta_with_locus.loc[duplicated_keys, ["assembly", "Locus Tag", "Gene", "Product"]]
        .sort_values(["assembly", "Locus Tag"])
        .head(20)
    )
    display(examples)
    raise ValueError(
        "Existem chaves assembly + Locus Tag duplicadas no Bakta. "
        "A correspondência é ambígua e não será resolvida automaticamente."
    )

print(f"Assemblies históricas: {len(target_assemblies)}")
print(f"Marcadores históricos: {len(historical_markers)}")
print(f"Marcadores finais: {len(markers)} (15 membrana/LPS + 46 esporulação)")
print(f"Orthogroups: {len(orthogroups_wide):,}")
print(f"Colunas de genomas no OrthoFinder: {len(of_assembly_columns)}")
print(f"Assemblies-alvo com pelo menos uma linha no Bakta: {bakta['assembly'].nunique()}/{len(target_assemblies)}")
print(f"Assemblies totais observadas no arquivo consolidado do Bakta: {len(all_bakta_assemblies)}")



## 3. Preparação das anotações Bakta

A correspondência direta usa somente o campo `Gene`, por igualdade exata com o nome do marcador. Não são aplicados aliases, busca textual em `Product` ou regras especiais por marcador.

Proteínas com chamada direta continuam válidas mesmo que não estejam em um orthogroup aprovado. Isso implementa explicitamente a regra:

> **presença final = Bakta positivo OU orthogroup aprovado**.


In [ ]:

bakta_proteins = (
    bakta_with_locus
    .rename(
        columns={
            "Locus Tag": "locus_tag",
            "Gene": "bakta_gene",
            "Product": "bakta_product",
            "Type": "bakta_type",
        }
    )
    [["assembly", "locus_tag", "bakta_gene", "bakta_product", "bakta_type"]]
    .copy()
)

direct_marker_proteins = (
    bakta_proteins.loc[bakta_proteins["bakta_gene"].isin(markers)]
    .rename(columns={"bakta_gene": "marker"})
    [["assembly", "locus_tag", "marker", "bakta_product", "bakta_type"]]
    .copy()
)

non_cds_marker_hits = direct_marker_proteins.loc[
    direct_marker_proteins["bakta_type"].ne("cds")
]
if not non_cds_marker_hits.empty:
    print("ATENÇÃO: foram encontradas chamadas de marcador fora de `Type == cds`.")
    display(non_cds_marker_hits.head(20))

direct_count_observed = (
    direct_marker_proteins
    .drop_duplicates(["assembly", "marker"])
    .groupby("marker")["assembly"]
    .nunique()
    .reindex(markers, fill_value=0)
)

reference_check = pd.DataFrame({
    "marker": NEW_MEMBRANE_MARKERS,
    "direct_positive_observed_in_supplied_bakta": [
        int(direct_count_observed[g]) for g in NEW_MEMBRANE_MARKERS
    ],
    "historical_reference_not_forced": [
        REFERENCE_DIRECT_COUNTS[g] for g in NEW_MEMBRANE_MARKERS
    ],
})
reference_check["difference_observed_minus_reference"] = (
    reference_check["direct_positive_observed_in_supplied_bakta"]
    - reference_check["historical_reference_not_forced"]
)

print("Chamadas diretas observadas para os três marcadores novos:")
display(reference_check)



## 4. Conversão dos orthogroups para formato longo

Cada célula não vazia da matriz do OrthoFinder pode conter uma ou várias proteínas separadas por vírgula. O formato esperado dos identificadores proteicos é validado explicitamente como:

`assembly|locus_tag`

Além disso, a parte `assembly` do identificador deve ser idêntica à coluna do OrthoFinder onde a proteína foi encontrada. Se isso não for verdade, o notebook interrompe esta etapa em vez de tentar adivinhar a correspondência.


In [ ]:

# `stack` evita materializar milhões de células vazias que apareceriam com `melt`.
og_indexed = orthogroups_wide.set_index("orthogroup")[of_assembly_columns]

try:
    stacked = og_indexed.stack(future_stack=True).dropna()
except TypeError:
    # Compatibilidade com versões mais antigas do pandas.
    stacked = og_indexed.stack(dropna=True)

og_cells_long = (
    stacked
    .rename("protein_field")
    .rename_axis(index=["orthogroup", "orthofinder_assembly"])
    .reset_index()
)

og_cells_long["protein_id"] = og_cells_long["protein_field"].str.split(",")
orthogroups_long = (
    og_cells_long
    .explode("protein_id", ignore_index=True)
    .drop(columns="protein_field")
)

orthogroups_long["protein_id"] = orthogroups_long["protein_id"].astype("string").str.strip()

parsed = orthogroups_long["protein_id"].str.extract(
    r"^(?P<protein_assembly>[^|]+)\|(?P<locus_tag>[^|]+)$"
)

parse_failed = parsed.isna().any(axis=1)
if parse_failed.any():
    examples = orthogroups_long.loc[parse_failed, "protein_id"].head(20).tolist()
    raise ValueError(
        "Identificadores proteicos incompatíveis com `assembly|locus_tag`. "
        f"Exemplos: {examples}"
    )

orthogroups_long = pd.concat([orthogroups_long, parsed], axis=1)

assembly_mismatch = (
    orthogroups_long["orthofinder_assembly"]
    .ne(orthogroups_long["protein_assembly"])
)
if assembly_mismatch.any():
    examples = orthogroups_long.loc[
        assembly_mismatch,
        ["orthogroup", "orthofinder_assembly", "protein_id", "protein_assembly"],
    ].head(20)
    display(examples)
    raise ValueError(
        "A assembly codificada no protein_id não coincide com a coluna do OrthoFinder."
    )

if orthogroups_long["protein_id"].duplicated().any():
    examples = orthogroups_long.loc[
        orthogroups_long["protein_id"].duplicated(keep=False),
        ["orthogroup", "orthofinder_assembly", "protein_id"],
    ].head(20)
    display(examples)
    raise ValueError(
        "O mesmo protein_id aparece mais de uma vez na matriz de orthogroups."
    )

orthogroups_long = orthogroups_long.rename(
    columns={"orthofinder_assembly": "assembly"}
)

print(f"Orthogroups preservados: {orthogroups_long['orthogroup'].nunique():,}")
print(f"Proteínas no OrthoFinder: {len(orthogroups_long):,}")
print(f"Assemblies representadas: {orthogroups_long['assembly'].nunique()}")
display(orthogroups_long.head())



## 5. Correspondência entre proteínas, Bakta e OrthoFinder

A correspondência é feita pela chave **exata** `(assembly, locus_tag)`.

A cobertura Bakta por assembly é auditada usando as proteínas do OrthoFinder como referência:

- `complete`: todas as proteínas do OrthoFinder daquela assembly encontram a mesma chave no Bakta;
- `partial`: há alguma correspondência, mas não para todas as proteínas;
- `missing`: nenhuma proteína do OrthoFinder encontra uma linha Bakta.

Somente `complete` permite emitir ausências (`0`) para aquela assembly. Positivos encontrados em uma assembly parcial continuam sendo preservados.


In [ ]:

bakta_for_merge = bakta_proteins.copy()
bakta_for_merge["_bakta_match"] = 1

protein_evidence = orthogroups_long.merge(
    bakta_for_merge,
    on=["assembly", "locus_tag"],
    how="left",
    validate="many_to_one",
)

protein_evidence["_bakta_match"] = (
    protein_evidence["_bakta_match"]
    .fillna(0)
    .astype("int8")
)

target_index = pd.Index(target_assemblies, name="assembly")

coverage_by_assembly = (
    protein_evidence
    .groupby("assembly", sort=False)
    .agg(
        n_orthofinder_proteins=("protein_id", "size"),
        n_bakta_matched_to_orthofinder=("_bakta_match", "sum"),
    )
    .reindex(target_index)
)

coverage_by_assembly["mapping_fraction"] = (
    coverage_by_assembly["n_bakta_matched_to_orthofinder"]
    / coverage_by_assembly["n_orthofinder_proteins"]
)

bakta_rows_by_assembly = bakta.groupby("assembly").size()
coverage_by_assembly["n_bakta_rows"] = (
    bakta_rows_by_assembly.reindex(target_index).fillna(0).astype(int)
)

coverage_by_assembly["status"] = np.select(
    [
        (
            coverage_by_assembly["n_orthofinder_proteins"].gt(0)
            & coverage_by_assembly["n_bakta_matched_to_orthofinder"]
              .eq(coverage_by_assembly["n_orthofinder_proteins"])
        ),
        coverage_by_assembly["n_bakta_matched_to_orthofinder"].eq(0),
    ],
    ["complete", "missing"],
    default="partial",
)

complete_bakta_assemblies = coverage_by_assembly.index[
    coverage_by_assembly["status"].eq("complete")
].tolist()

# Chamada direta: 0 somente nas assemblies completas; 1 sempre que há evidência.
direct_matrix = pd.DataFrame(
    pd.NA,
    index=target_index,
    columns=markers,
    dtype="Int64",
)
direct_matrix.loc[complete_bakta_assemblies, :] = 0

for row in (
    direct_marker_proteins[["assembly", "marker"]]
    .drop_duplicates()
    .itertuples(index=False)
):
    direct_matrix.loc[row.assembly, row.marker] = 1

# Proteínas diretamente anotadas que não aparecem no OrthoFinder.
direct_to_og = direct_marker_proteins.merge(
    orthogroups_long[["assembly", "locus_tag", "protein_id", "orthogroup"]],
    on=["assembly", "locus_tag"],
    how="left",
    validate="one_to_one",
)

direct_not_in_orthofinder = direct_to_og.loc[
    direct_to_og["protein_id"].isna()
].copy()

print("Status de cobertura Bakta:")
print(coverage_by_assembly["status"].value_counts().to_string())

problem_coverage = coverage_by_assembly.loc[
    coverage_by_assembly["status"].ne("complete")
]
if not problem_coverage.empty:
    print(
        "\nATENÇÃO: a cobertura Bakta não é completa para todas as 229 assemblies. "
        "Essas linhas bloquearão ausências binárias até que a entrada seja corrigida."
    )
    display(problem_coverage)

print(
    "\nProteínas com marcador direto que não aparecem no OrthoFinder "
    f"(a chamada Bakta é preservada): {len(direct_not_in_orthofinder)}"
)



## 6. Identificação e aprovação automática dos orthogroups candidatos

Uma proteína Bakta anotada exatamente com um marcador é uma **semente** daquele marcador no seu orthogroup.

Para cada par marcador–orthogroup candidato:

- `n_seed` = número de proteínas distintas do orthogroup anotadas pelo Bakta com aquele marcador;
- `n_proteins` = número total de proteínas distintas do orthogroup no conjunto completo de 229 genomas;
- `seed_fraction = n_seed / n_proteins`.

O denominador é calculado **a partir do OrthoFinder**, não apenas das proteínas que conseguiram ser associadas ao Bakta. Isso evita inflar artificialmente a fração de concordância quando há anotações ausentes.

A decisão é exclusivamente automática:

- `accept` se `seed_fraction >= 0.50` **e** `n_seed >= 2`;
- `reject` caso contrário.


In [ ]:

seed = (
    protein_evidence.loc[protein_evidence["bakta_gene"].isin(markers)]
    [["protein_id", "assembly", "orthogroup", "locus_tag", "bakta_gene"]]
    .copy()
)
seed["marker"] = seed["bakta_gene"].astype("string")

og_size = (
    orthogroups_long
    .groupby("orthogroup")["protein_id"]
    .nunique()
    .rename("n_proteins")
)

og_genomes = (
    orthogroups_long
    .groupby("orthogroup")["assembly"]
    .nunique()
    .rename("n_genomes")
)

seed_stats = (
    seed
    .groupby(["marker", "orthogroup"])
    .agg(
        n_seed=("protein_id", "nunique"),
        n_seed_genomes=("assembly", "nunique"),
    )
    .reset_index()
)

orthogroup_review_auto = (
    seed_stats
    .merge(og_size.reset_index(), on="orthogroup", validate="many_to_one")
    .merge(og_genomes.reset_index(), on="orthogroup", validate="many_to_one")
)

orthogroup_review_auto["seed_fraction"] = (
    orthogroup_review_auto["n_seed"]
    / orthogroup_review_auto["n_proteins"]
)

# Resumo legível das anotações de gene observadas no OG entre proteínas
# que possuem uma linha Bakta associada.
gene_annotations_by_og = (
    protein_evidence
    .dropna(subset=["bakta_gene"])
    .groupby("orthogroup")["bakta_gene"]
    .agg(
        lambda s: "; ".join(
            f"{gene}:{count}"
            for gene, count in s.value_counts().sort_index().items()
        )
    )
    .rename("bakta_gene_counts_in_orthogroup")
    .reset_index()
)

orthogroup_review_auto = orthogroup_review_auto.merge(
    gene_annotations_by_og,
    on="orthogroup",
    how="left",
    validate="many_to_one",
)

passes_fraction = orthogroup_review_auto["seed_fraction"].ge(MIN_SEED_FRACTION)
passes_seeds = orthogroup_review_auto["n_seed"].ge(MIN_SEEDS)

orthogroup_review_auto["decision"] = np.where(
    passes_fraction & passes_seeds,
    "accept",
    "reject",
)

orthogroup_review_auto["reason"] = np.select(
    [
        passes_fraction & passes_seeds,
        (~passes_fraction) & (~passes_seeds),
        ~passes_fraction,
        ~passes_seeds,
    ],
    [
        "accept: seed_fraction>=0.50 and n_seed>=2",
        "reject: seed_fraction<0.50 and n_seed<2",
        "reject: seed_fraction<0.50",
        "reject: n_seed<2",
    ],
    default="reject",
)

orthogroup_review_auto["marker_category"] = (
    orthogroup_review_auto["marker"].map(marker_category)
)

orthogroup_review_auto = (
    orthogroup_review_auto[
        [
            "marker",
            "marker_category",
            "orthogroup",
            "n_genomes",
            "n_proteins",
            "n_seed",
            "n_seed_genomes",
            "seed_fraction",
            "decision",
            "reason",
            "bakta_gene_counts_in_orthogroup",
        ]
    ]
    .sort_values(
        ["marker_category", "marker", "decision", "seed_fraction", "orthogroup"],
        ascending=[True, True, True, False, True],
    )
    .reset_index(drop=True)
)

accepted_orthogroups = (
    orthogroup_review_auto.loc[
        orthogroup_review_auto["decision"].eq("accept"),
        ["marker", "orthogroup"],
    ]
    .drop_duplicates()
)

markers_with_candidates = set(orthogroup_review_auto["marker"])
markers_with_accepted_og = set(accepted_orthogroups["marker"])

print(
    f"Marcadores com pelo menos um OG candidato: "
    f"{len(markers_with_candidates)}/{len(markers)}"
)
print(
    f"Marcadores com pelo menos um OG aprovado: "
    f"{len(markers_with_accepted_og)}/{len(markers)}"
)
print(
    "Sem candidato:",
    [m for m in markers if m not in markers_with_candidates],
)
print(
    "Sem OG aprovado:",
    [m for m in markers if m not in markers_with_accepted_og],
)

display(orthogroup_review_auto.head(30))



## 7. Construção das matrizes de evidência

São construídas duas matrizes separadas:

- `direct_matrix`: evidência direta do Bakta;
- `inferred_matrix`: presença em um orthogroup aprovado.

A matriz do OrthoFinder cobre todas as 229 assemblies, portanto `inferred_matrix` é uma matriz de **evidência** 0/1: `1` significa que existe pelo menos uma proteína em um OG aprovado; `0` significa apenas que não há evidência por OG aprovado.

A tabela `gene_evidence_accepted.tsv` mantém a proveniência no nível de proteína.


In [ ]:

accepted_members = accepted_orthogroups.merge(
    protein_evidence[
        [
            "orthogroup",
            "assembly",
            "protein_id",
            "locus_tag",
            "bakta_gene",
            "bakta_product",
            "bakta_type",
            "_bakta_match",
        ]
    ],
    on="orthogroup",
    how="inner",
    validate="one_to_many",
)

accepted_members = accepted_members.rename(
    columns={"_bakta_match": "has_matching_bakta_row"}
)

accepted_members["marker_category"] = accepted_members["marker"].map(marker_category)
accepted_members = accepted_members[
    [
        "marker",
        "marker_category",
        "orthogroup",
        "assembly",
        "protein_id",
        "locus_tag",
        "has_matching_bakta_row",
        "bakta_gene",
        "bakta_product",
        "bakta_type",
    ]
].sort_values(["marker", "orthogroup", "assembly", "protein_id"])

inferred_matrix = pd.DataFrame(
    0,
    index=target_index,
    columns=markers,
    dtype="Int64",
)

inferred_presence = (
    accepted_members[["assembly", "marker"]]
    .drop_duplicates()
)

for row in inferred_presence.itertuples(index=False):
    inferred_matrix.loc[row.assembly, row.marker] = 1

# Evidência agregada por chamada inferida.
inferred_evidence_by_call = (
    accepted_members
    .groupby(["assembly", "marker"], as_index=False)
    .agg(
        orthogroups=(
            "orthogroup",
            lambda s: ", ".join(sorted(set(s.astype(str)))),
        ),
        n_orthogroup_proteins=("protein_id", "nunique"),
        protein_ids=(
            "protein_id",
            lambda s: ", ".join(sorted(set(s.astype(str)))),
        ),
    )
)

direct_values_for_inferred_calls = []
for row in inferred_evidence_by_call.itertuples(index=False):
    direct_values_for_inferred_calls.append(
        direct_matrix.loc[row.assembly, row.marker]
    )

inferred_evidence_by_call["bakta_direct"] = pd.array(
    direct_values_for_inferred_calls,
    dtype="Int64",
)

# "Adicional" só é declarado quando o Bakta foi efetivamente avaliado
# para aquela célula e retornou 0.
additional_gene_calls = inferred_evidence_by_call.loc[
    inferred_evidence_by_call["bakta_direct"].eq(0).fillna(False)
].copy()

additional_gene_calls["orthogroup_inferred"] = 1
additional_gene_calls = additional_gene_calls[
    [
        "assembly",
        "marker",
        "bakta_direct",
        "orthogroup_inferred",
        "orthogroups",
        "n_orthogroup_proteins",
        "protein_ids",
    ]
].sort_values(["marker", "assembly"])

print(f"Proteínas associadas a OGs aprovados: {len(accepted_members):,}")
print(f"Chamadas assembly–marcador inferidas: {len(inferred_evidence_by_call):,}")
print(
    "Chamadas realmente adicionais em relação a um Bakta avaliado como 0:",
    len(additional_gene_calls),
)



## 8. Construção da matriz híbrida final

A chamada híbrida preserva qualquer positivo:

`hybrid = Bakta positivo OR orthogroup aprovado`

Para assemblies com cobertura Bakta `complete`, a ausência dos dois tipos de evidência é registrada como `0`.

Para assemblies `partial` ou `missing`:

- qualquer positivo direto ou inferido é registrado como `1`;
- a falta de evidência positiva permanece `NA`, pois o Bakta não foi avaliado de forma completa.

Esse comportamento é deliberado e impede que lacunas de entrada sejam transformadas em ausências.


In [ ]:

hybrid_matrix = pd.DataFrame(
    pd.NA,
    index=target_index,
    columns=markers,
    dtype="Int64",
)

# Assemblies com Bakta completo: 0/1 pode ser resolvido de forma estrita.
direct_complete = direct_matrix.loc[complete_bakta_assemblies].astype(int)
inferred_complete = inferred_matrix.loc[complete_bakta_assemblies].astype(int)

hybrid_matrix.loc[complete_bakta_assemblies] = (
    direct_complete.eq(1) | inferred_complete.eq(1)
).astype(int)

# Um positivo é informativo mesmo se a cobertura total daquela assembly for parcial.
positive_anywhere = (
    direct_matrix.eq(1).fillna(False)
    | inferred_matrix.eq(1)
)
hybrid_matrix = hybrid_matrix.mask(positive_anywhere, 1)

n_unresolved = int(hybrid_matrix.isna().sum().sum())
assemblies_with_unresolved = hybrid_matrix.index[
    hybrid_matrix.isna().any(axis=1)
].tolist()

print(f"Células resolvidas: {hybrid_matrix.notna().sum().sum():,}")
print(f"Células não resolvidas (NA): {n_unresolved:,}")
print(f"Assemblies com pelo menos um NA: {len(assemblies_with_unresolved)}")

if assemblies_with_unresolved:
    print("Assemblies ainda não totalmente resolvidas:")
    print(assemblies_with_unresolved)

display(hybrid_matrix.head())



## 9. Comparação com a matriz que usa apenas o Bakta e com referências anteriores

A comparação histórica é diagnóstica; ela não altera as chamadas novas.

- Para os 58 marcadores antigos, compara-se somente onde a nova matriz possui um estado resolvido.
- A matriz final anterior, se fornecida, é tratada como um arquivo opcional de comparação.
- Os valores de referência `bamA=28`, `lptA=11` e `ompH=21` são mostrados ao lado das chamadas diretas observadas, mas **não são usados como alvo nem como restrição**.


In [ ]:

historical_binary = (
    historical
    .set_index("assembly")[historical_markers]
    .astype("Int64")
    .reindex(target_index)
)

previous_final = None
if PREVIOUS_FINAL_PATH.exists():
    previous_final = pd.read_csv(PREVIOUS_FINAL_PATH, sep="\t")
    if "assembly" not in previous_final.columns:
        raise ValueError(
            f"{PREVIOUS_FINAL_PATH.name} existe, mas não possui a coluna `assembly`."
        )
    if previous_final["assembly"].duplicated().any():
        raise ValueError(
            f"{PREVIOUS_FINAL_PATH.name} possui assemblies duplicadas."
        )
    previous_final["assembly"] = previous_final["assembly"].astype("string").str.strip()

    if set(previous_final["assembly"]) != target_set:
        print(
            "ATENÇÃO: a matriz final anterior não tem exatamente o mesmo conjunto "
            "de assemblies e será comparada apenas na interseção."
        )

summary_rows = []

for marker in markers:
    direct_positive = int(direct_matrix[marker].eq(1).sum())
    inferred_positive = int(inferred_matrix[marker].eq(1).sum())
    additional_positive = int(
        additional_gene_calls["marker"].eq(marker).sum()
    )
    hybrid_positive = int(hybrid_matrix[marker].eq(1).sum())
    unresolved = int(hybrid_matrix[marker].isna().sum())
    resolved = len(target_index) - unresolved

    historical_positive = pd.NA
    changed_vs_historical_on_resolved = pd.NA

    if marker in historical_binary.columns:
        historical_positive = int(historical_binary[marker].sum())

        comparable = hybrid_matrix[marker].notna()
        changed_vs_historical_on_resolved = int(
            hybrid_matrix.loc[comparable, marker]
            .astype(int)
            .ne(historical_binary.loc[comparable, marker].astype(int))
            .sum()
        )

    previous_positive = pd.NA
    changed_vs_previous_on_resolved = pd.NA

    if (
        previous_final is not None
        and marker in previous_final.columns
    ):
        prev_series = (
            previous_final
            .set_index("assembly")[marker]
            .reindex(target_index)
        )

        valid_prev = prev_series.isin([0, 1])
        comparable_prev = hybrid_matrix[marker].notna() & valid_prev

        if valid_prev.any():
            previous_positive = int(prev_series.loc[valid_prev].astype(int).sum())

        if comparable_prev.any():
            changed_vs_previous_on_resolved = int(
                hybrid_matrix.loc[comparable_prev, marker]
                .astype(int)
                .ne(prev_series.loc[comparable_prev].astype(int))
                .sum()
            )

    summary_rows.append({
        "marker": marker,
        "category": marker_category[marker],
        "direct_positive_observed": direct_positive,
        "orthogroup_inferred_positive": inferred_positive,
        "additional_positive_vs_evaluable_bakta": additional_positive,
        "hybrid_positive_observed": hybrid_positive,
        "resolved_cells": resolved,
        "unresolved_cells": unresolved,
        "historical_positive_58": historical_positive,
        "changed_vs_historical_on_resolved": changed_vs_historical_on_resolved,
        "previous_final_positive_if_available": previous_positive,
        "changed_vs_previous_on_resolved": changed_vs_previous_on_resolved,
        "direct_reference_if_provided": REFERENCE_DIRECT_COUNTS.get(marker, pd.NA),
    })

marker_summary = pd.DataFrame(summary_rows)

marker_summary["direct_observed_minus_reference"] = pd.NA
for marker, reference in REFERENCE_DIRECT_COUNTS.items():
    idx = marker_summary["marker"].eq(marker)
    marker_summary.loc[idx, "direct_observed_minus_reference"] = (
        marker_summary.loc[idx, "direct_positive_observed"].astype(int)
        - reference
    )

print("Comparação dos três marcadores novos com as referências fornecidas:")
display(
    marker_summary.loc[
        marker_summary["marker"].isin(NEW_MEMBRANE_MARKERS),
        [
            "marker",
            "direct_positive_observed",
            "direct_reference_if_provided",
            "direct_observed_minus_reference",
            "unresolved_cells",
        ],
    ]
)

display(marker_summary)


## 10. Validações finais

A exportação das **duas matrizes finais** só ocorre após conferir as 229 assemblies,
61 marcadores (15 de membrana/LPS e 46 de esporulação), cobertura Bakta completa,
ausência de `NA` e estados exclusivamente binários.

As colunas auxiliares são mantidas **sem alteração**:

- **`Spore former`**: classificação de Galperin (`1` = esporulante, `2` = não esporulante,
  `0` = sem classificação); não é uma previsão calculada a partir dos genes.
- **`expected`**: classificação de envelope **baseada na literatura** (diderm/monoderm),
  não recalculada a partir dos marcadores.
- **`n_om` e `observed`**: informações históricas da tabela original, que podem não
  refletir os 15 marcadores de membrana/LPS usados agora.
- **`label` e `assembly`**: identificadores para ligar a matriz às filogenias.

Os diagnósticos continuam visíveis no próprio notebook, mesmo quando não são exportados.

In [ ]:

validation_rows = []

def add_validation(check, passed, detail):
    validation_rows.append({
        "check": check,
        "passed": bool(passed),
        "detail": detail,
    })

add_validation(
    "229 assemblies na matriz histórica",
    len(target_index) == EXPECTED_N_ASSEMBLIES,
    f"observado={len(target_index)}; esperado={EXPECTED_N_ASSEMBLIES}",
)
add_validation(
    "229 assemblies no OrthoFinder",
    orthogroups_long["assembly"].nunique() == EXPECTED_N_ASSEMBLIES,
    f"observado={orthogroups_long['assembly'].nunique()}; esperado={EXPECTED_N_ASSEMBLIES}",
)
add_validation(
    "61 marcadores únicos",
    len(markers) == EXPECTED_N_MARKERS and len(set(markers)) == EXPECTED_N_MARKERS,
    f"observado={len(markers)}; esperado={EXPECTED_N_MARKERS}",
)
add_validation(
    "15 marcadores de membrana/LPS",
    sum(marker_category[m] == "outer_membrane_LPS" for m in markers) == 15,
    f"observado={sum(marker_category[m] == 'outer_membrane_LPS' for m in markers)}; esperado=15",
)
add_validation(
    "46 marcadores de esporulação",
    sum(marker_category[m] == "sporulation" for m in markers) == 46,
    f"observado={sum(marker_category[m] == 'sporulation' for m in markers)}; esperado=46",
)
add_validation(
    "Cobertura Bakta completa nas 229 assemblies",
    coverage_by_assembly["status"].eq("complete").all(),
    coverage_by_assembly["status"].value_counts().to_dict(),
)
add_validation(
    "Matriz híbrida sem NA",
    not hybrid_matrix.isna().any().any(),
    f"n_unresolved={int(hybrid_matrix.isna().sum().sum())}",
)

resolved_values = set(
    hybrid_matrix.stack().dropna().astype(int).unique().tolist()
)
add_validation(
    "Estados resolvidos estritamente binários",
    resolved_values.issubset({0, 1}),
    f"valores_resolvidos={sorted(resolved_values)}",
)

validation_report = pd.DataFrame(validation_rows)
display(validation_report)

ready_for_binary_export = bool(
    validation_report["passed"].all()
)

print("Pronto para exportação binária final:", ready_for_binary_export)


## 11. Exportação enxuta

**Execução aprovada — cinco TSVs:**

1. `gene_presence_absence_final_61.tsv`: matriz binária (61 genes × 229 genomas).
2. `heatmap_matrix_final_61.tsv`: matriz com os mesmos 61 genes e seis colunas
   auxiliares históricas, incluindo o `Spore former` de Galperin e o `expected`
   baseado na literatura.
3. `orthogroup_review_auto.tsv`: critérios e decisão dos OGs candidatos.
4. `marker_summary.tsv`: contagens por gene e diferenças em relação às referências.
5. `additional_gene_calls.tsv`: genoma × marcador adicionado pelos OGs, com a evidência
   necessária para identificar as presenças adicionais no heatmap.

As tabelas `accepted_members`, `coverage_by_assembly` e `validation_report`
continuam disponíveis como variáveis do notebook, mas não são exportadas normalmente.

**Se houver falhas**, o notebook *não* salva as duas matrizes finais; no lugar delas
exporta arquivos de diagnóstico e uma matriz parcial com `NA`. Ele também interrompe
com uma mensagem explicando o impedimento.

Arquivos exportados por versões anteriores no mesmo diretório não são apagados
automaticamente. Os nomes impressos ao final identificam somente os arquivos desta execução.

In [ ]:
# -------------------------------------------------------------
# Preparar um diagnóstico somente para execuções com problemas.
# Não há necessidade de exportá-lo quando tudo está validado.
# -------------------------------------------------------------
issue_rows = []

for assembly, row in coverage_by_assembly.loc[
    coverage_by_assembly["status"].ne("complete")
].iterrows():
    issue_rows.append({
        "severity": "blocking",
        "issue_type": (
            "bakta_assembly_missing" if row["status"] == "missing"
            else "bakta_orthofinder_partial_mapping"
        ),
        "assembly": assembly,
        "marker": pd.NA,
        "protein_id": pd.NA,
        "locus_tag": pd.NA,
        "detail": (
            f"status={row['status']}; "
            f"OF_proteins={int(row['n_orthofinder_proteins'])}; "
            f"Bakta_matches={int(row['n_bakta_matched_to_orthofinder'])}; "
            f"mapping_fraction={row['mapping_fraction']:.6f}; "
            f"Bakta_rows={int(row['n_bakta_rows'])}"
        ),
    })

for row in direct_not_in_orthofinder.itertuples(index=False):
    issue_rows.append({
        "severity": "informational",
        "issue_type": "direct_marker_protein_not_in_orthofinder",
        "assembly": row.assembly,
        "marker": row.marker,
        "protein_id": pd.NA,
        "locus_tag": row.locus_tag,
        "detail": "Presença direta preservada; proteína não localizada nos Orthogroups.",
    })

identifier_inconsistencies = pd.DataFrame(
    issue_rows,
    columns=[
        "severity", "issue_type", "assembly", "marker",
        "protein_id", "locus_tag", "detail",
    ],
)

# -------------------------------------------------------------
# Os cinco arquivos principais só são escritos após passar
# todas as validações. Criamos e verificamos ambos os DataFrames
# antes de iniciar qualquer exportação.
# -------------------------------------------------------------
if ready_for_binary_export:
    final_presence = hybrid_matrix.astype("int64").reset_index()
    if final_presence.shape != (EXPECTED_N_ASSEMBLIES, EXPECTED_N_MARKERS + 1):
        raise AssertionError(f"Forma inesperada da matriz final: {final_presence.shape}")

    heatmap_final = historical[AUX_COLUMNS].merge(
        final_presence,
        on="assembly",
        how="left",
        validate="one_to_one",
    )[AUX_COLUMNS + markers]

    if heatmap_final.shape != (
        EXPECTED_N_ASSEMBLIES, len(AUX_COLUMNS) + EXPECTED_N_MARKERS
    ):
        raise AssertionError(f"Forma inesperada do heatmap: {heatmap_final.shape}")

    # O merge pode mudar o dtype de assembly de `string` para `object`.
    # Restauramos o dtype para validar equivalência sem falsos alarmes.
    heatmap_final["assembly"] = heatmap_final["assembly"].astype(
        historical["assembly"].dtype
    )
    # As colunas auxiliares vêm da tabela histórica sem alterações.
    assert heatmap_final[AUX_COLUMNS].equals(historical[AUX_COLUMNS]), (
        "As informações auxiliares deixaram de corresponder à matriz histórica."
    )

    exports = {
        "gene_presence_absence_final_61.tsv": final_presence,
        "heatmap_matrix_final_61.tsv": heatmap_final,
        "orthogroup_review_auto.tsv": orthogroup_review_auto,
        "marker_summary.tsv": marker_summary,
        "additional_gene_calls.tsv": additional_gene_calls,
    }
    for filename, table in exports.items():
        table.to_csv(OUTPUT_DIR / filename, sep="\t", index=False)

    print(f"Curadoria validada: {len(exports)} TSVs exportados em {OUTPUT_DIR.resolve()}")
    for filename in exports:
        print(" -", filename)
    print("Informações externas preservadas: Spore former (Galperin), expected (literatura).")

else:
    # Diagnósticos não substituem matrizes finais binárias.
    diagnostics = {
        "gene_presence_absence_partial_61_with_NA.tsv": hybrid_matrix.reset_index(),
        "input_coverage_by_assembly.tsv": coverage_by_assembly.reset_index(),
        "identifier_inconsistencies.tsv": identifier_inconsistencies,
    }
    for filename, table in diagnostics.items():
        table.to_csv(OUTPUT_DIR / filename, sep="\t", index=False)

    print("Exportação final BLOQUEADA. Arquivos de diagnóstico:")
    for filename in diagnostics:
        print(" -", filename)
    raise RuntimeError(
        "Curadoria incompleta: não foi exportada uma matriz binária final. "
        "Consulte as validações da seção 10 e os diagnósticos acima."
    )

## 12. Resumo metodológico e uso posterior

O procedimento combina anotações diretas do **Bakta** com homólogos pertencentes
àqueles orthogroups do **OrthoFinder** que apresentem concordância de anotação
`n_seed / n_proteins >= 0.50` e pelo menos duas proteínas-semente.
A matriz híbrida usa a regra **Bakta positivo OU orthogroup aprovado**.

Trata-se de uma aproximação didática e reprodutível: homologia não demonstra
função experimental, e um conjunto de genes não prova um fenótipo.

Na tabela enriquecida para visualização:

- `Spore former` reproduz a **classificação fenotípica de Galperin**
  (`1` esporulante, `2` não esporulante, `0` não classificado);
- `expected` preserva o **tipo de envelope descrito na literatura**;
- `n_om` e `observed` permanecem como informações legadas da tabela histórica;
- nenhuma das quatro colunas é recalculada durante a curadoria.

Quando todas as verificações passam, utilize `gene_presence_absence_final_61.tsv`
para Fritz's D e `heatmap_matrix_final_61.tsv` para os heatmaps. As presenças
acrescentadas pelo OrthoFinder também ficam listadas em `additional_gene_calls.tsv`.

**Atenção:** arquivos TSV antigos que tenham ficado no diretório de resultados
após execuções de versões anteriores não são apagados por este notebook.